# 🎬 WAN 2.1 (1.3B / 14B GGUF) trên Google Colab Free với ComfyUI
### Bộ công cụ 1-Click tối ưu hóa cho YouTube Faceless, Shorts, TikTok & Chống tràn RAM (OOM)

Notebook này thiết lập trọn gói môi trường để tạo video AI điện ảnh với mô hình **Wan 2.1** (Image-to-Video, Text-to-Video, First Frame - Last Frame):
- ⚡ **Tải Model tốc độ cao:** Dùng `aria2` đa luồng (~1-2 phút cho bộ model).
- 🛡️ **Chống tràn RAM/VRAM:** Tích hợp Text Encoder UMT5 FP8 Scaled, Tiled VAE, và hỗ trợ 14B GGUF.
- 💾 **Google Drive Auto-Sync:** Tự động tạo Symlink lưu thẳng toàn bộ video đã render vào Drive cá nhân (`MyDrive/Wan21_Videos`).
- 🌐 **Cloudflare Tunnel:** Cung cấp URL trực tiếp truy cập ComfyUI bảo mật, không cần ngrok token.

## Bước 1: Cài đặt ComfyUI & Các Custom Nodes cần thiết

In [ ]:
#@title 1. Cài đặt ComfyUI & Tiện ích cần thiết { display-mode: "form" }
import os
import subprocess

print("🔍 Đang kiểm tra GPU...")
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

print("\n📦 1. Cài đặt aria2...")
!apt-get update -qq && apt-get install -y -qq aria2

print("\n🚀 2. Clone ComfyUI Core...")
if not os.path.exists('/content/ComfyUI'):
    !git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
    %cd /content/ComfyUI
    !pip install -q -r requirements.txt
    !pip install -q xformers==0.0.28.post1 --index-url https://download.pytorch.org/whl/cu121

print("\n🧩 3. Cài đặt Custom Nodes tối ưu cho Wan 2.1...")
%cd /content/ComfyUI/custom_nodes
for repo in [
    'https://github.com/ltdrdata/ComfyUI-Manager.git',
    'https://github.com/kijai/ComfyUI-WanVideoWrapper.git',
    'https://github.com/city96/ComfyUI-GGUF.git',
    'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git'
]:
    name = repo.split('/')[-1].replace('.git', '')
    if not os.path.exists(name):
        subprocess.run(f'git clone {repo}', shell=True, check=True)
        req = os.path.join(name, 'requirements.txt')
        if os.path.exists(req):
            subprocess.run(f'pip install -q -r {req}', shell=True, check=True)

print("\n✅ Hoàn tất cài đặt ComfyUI và các custom node cần thiết!")

## Bước 2: Tải Models Wan 2.1 (Tùy chọn 1.3B hoặc 14B GGUF)

In [ ]:
#@title 2. Tải Trọng số Wan 2.1 (Tùy chọn tải 1.3B hoặc 14B GGUF) { display-mode: "form" }
import os
import subprocess

download_wan_1_3B = True #@param {type:"boolean"}
download_wan_14B_GGUF = False #@param {type:"boolean"}

%cd /content/ComfyUI

def download_aria(url, out_dir, filename):
    os.makedirs(out_dir, exist_ok=True)
    target_path = os.path.join(out_dir, filename)
    if not os.path.exists(target_path):
        print(f"⏳ Đang tải {filename} vào {out_dir}...")
        cmd = f'aria2c --console-log-level=error -c -x 16 -s 16 -k 1M --allow-overwrite=true "{url}" -d "{out_dir}" -o "{filename}"'
        try:
            subprocess.run(cmd, shell=True, check=True)
            print(f"✅ Đã tải xong {filename}")
        except Exception:
            print(f"⚠️ Aria2c gặp sự cố, tự động chuyển sang tải bằng wget an toàn...")
            subprocess.run(f'wget -c "{url}" -O "{target_path}"', shell=True, check=True)
            print(f"✅ Đã tải xong {filename} qua wget")
    else:
        print(f"⚡ File {filename} đã có sẵn!")

print("📥 1. Tải các thành phần cốt lõi chung (Text Encoder FP8 Scaled, VAE, CLIP Vision)...")
# Text Encoder UMT5 FP8 Scaled
download_aria(
    "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
    "/content/ComfyUI/models/text_encoders",
    "umt5_xxl_fp8_e4m3fn_scaled.safetensors"
)
# Wan 2.1 VAE
download_aria(
    "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors",
    "/content/ComfyUI/models/vae",
    "wan_2.1_vae.safetensors"
)
# CLIP Vision Model (Dành cho Image-to-Video)
download_aria(
    "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/clip_vision/clip_vision_h.safetensors",
    "/content/ComfyUI/models/clip_vision",
    "clip_vision_h.safetensors"
)

if download_wan_1_3B:
    print("\n📥 2. Tải bộ mô hình Wan 2.1 1.3B (Nhanh, nhẹ, tối ưu Colab T4)...")
    download_aria(
        "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/diffusion_models/wan2.1_i2v_480p_1.3B_bf16.safetensors",
        "/content/ComfyUI/models/diffusion_models",
        "wan2.1_i2v_480p_1.3B_bf16.safetensors"
    )
    download_aria(
        "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/diffusion_models/wan2.1_t2v_1.3B_bf16.safetensors",
        "/content/ComfyUI/models/diffusion_models",
        "wan2.1_t2v_1.3B_bf16.safetensors"
    )

if download_wan_14B_GGUF:
    print("\n📥 3. Tải mô hình Wan 2.1 14B GGUF Q4_K_M (Chất lượng điện ảnh đỉnh cao)...")
    download_aria(
        "https://huggingface.co/city96/Wan2.1-I2V-14B-480P-GGUF/resolve/main/Wan2.1-I2V-14B-480P_Q4_K_M.gguf",
        "/content/ComfyUI/models/unet",
        "Wan2.1-I2V-14B-480P_Q4_K_M.gguf"
    )
    download_aria(
        "https://huggingface.co/city96/Wan2.1-T2V-14B-GGUF/resolve/main/Wan2.1-T2V-14B_Q4_K_M.gguf",
        "/content/ComfyUI/models/unet",
        "Wan2.1-T2V-14B_Q4_K_M.gguf"
    )

print("\n🎉 TẤT CẢ MODEL ĐƯỢC CHỌN ĐÃ NẠP SẴN SÀNG!")

## Bước 3: Tự động kết nối Google Drive (Auto-Sync Output)

In [ ]:
#@title 3. (Khuyên dùng) Tự động kết nối Google Drive để lưu video vĩnh viễn { display-mode: "form" }
from google.colab import drive
import os, shutil

save_to_drive = True #@param {type:"boolean"}

if save_to_drive:
    print("📂 Đang kết nối Google Drive...")
    drive.mount('/content/drive')
    drive_output = '/content/drive/MyDrive/Wan21_Videos'
    comfy_output = '/content/ComfyUI/output'
    os.makedirs(drive_output, exist_ok=True)
    
    if os.path.exists(comfy_output) and not os.path.islink(comfy_output):
        shutil.rmtree(comfy_output, ignore_errors=True)
    if not os.path.exists(comfy_output):
        os.symlink(drive_output, comfy_output)
        
    print(f"✅ Video tạo ra sẽ tự động lưu vĩnh viễn vào: {drive_output}")
else:
    print("ℹ️ Video sẽ chỉ lưu tạm ở /content/ComfyUI/output")

#@title 4. KHỞI CHẠY COMFYUI & LẤY ĐƯỜNG LINK TRUY CẬP WEB { display-mode: "form" }
import subprocess, time, re

# 1. Cài đặt cloudflared
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# 2. Chạy ComfyUI với Legacy Frontend (Load siêu tốc, không bao giờ kẹt splash screen)
comfy_cmd = "python /content/ComfyUI/main.py --listen 127.0.0.1 --port 8188 --fp8_e4m3fn-text-enc --preview-method auto --enable-cors-header --front-end-version Comfy-Org/ComfyUI_legacy_frontend@latest"
comfy_proc = subprocess.Popen(comfy_cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# 3. Khởi động Cloudflare Tunnel
tunnel_cmd = "cloudflared tunnel --url http://127.0.0.1:8188"
tunnel_proc = subprocess.Popen(tunnel_cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

print("⏳ Đang khởi động ComfyUI và tạo đường dẫn công khai Cloudflare...")
tunnel_url = None
start_time = time.time()
while time.time() - start_time < 60:
    line = tunnel_proc.stdout.readline()
    if "trycloudflare.com" in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            break
    time.sleep(0.5)

if tunnel_url:
    print("\n=========================================================================")
    print(f"🔗 BẤM VÀO ĐÂY ĐỂ MỞ COMFYUI:  {tunnel_url}")
    print("=========================================================================\n")
    print("👉 Hướng dẫn: Mở link trên -> Kéo thả file Workflow .json vào màn hình để bắt đầu tạo video!")
else:
    print("⚠️ Chưa bắt được link tự động, đang theo dõi log tunnel:")

for line in comfy_proc.stdout:
    print(line, end='')

In [ ]:
#@title 4. KHỞI CHẠY COMFYUI & LẤY ĐƯỜNG LINK TRUY CẬP WEB { display-mode: "form" }
import subprocess, time, re

# 1. Cài đặt cloudflared
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# 2. Chạy ComfyUI và Cloudflare Tunnel
comfy_proc = subprocess.Popen(
    "python /content/ComfyUI/main.py --listen 127.0.0.1 --port 8188 --fp8_e4m3fn-text-enc --preview-method auto",
    shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
tunnel_proc = subprocess.Popen(
    "cloudflared tunnel --url http://127.0.0.1:8188",
    shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

print("⏳ Đang khởi động ComfyUI và tạo đường dẫn công khai Cloudflare...")
tunnel_url = None
start_time = time.time()
while time.time() - start_time < 60:
    line = tunnel_proc.stdout.readline()
    if "trycloudflare.com" in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            break
    time.sleep(0.5)

if tunnel_url:
    print("\n=========================================================================")
    print(f"🔗 BẤM VÀO ĐÂY ĐỂ MỞ COMFYUI:  {tunnel_url}")
    print("=========================================================================\n")
    print("👉 Hướng dẫn: Mở link trên -> Kéo thả file Workflow .json vào màn hình để bắt đầu tạo video!")
else:
    print("⚠️ Chưa bắt được link tự động, đang theo dõi log tunnel:")

for line in comfy_proc.stdout:
    print(line, end='')